# CGC AMPR Notebook

Documentation and manual test notebook for the CGC `AMPR` wrapper.

Structure:
- one simple end-to-end sequence
- one connected-session section for wrapper methods that do not require `initialize()`
- one initialized-session section for operational AMPR methods
- one low-level transport section for `open_port()` / `close_port()` style calls

Run the notebook on Windows with real hardware.


In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
for candidate in (current, *current.parents):
    if (candidate / "src").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the repository root containing src/.")

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Repo root: {REPO_ROOT}")
print(f"Using source tree: {SRC_DIR}")


In [ ]:
import time

from cgc.ampr import AMPR

DEVICE_ID = "AMPR_B"
COM_PORT = 5
BAUDRATE = 230400
HK_INTERVAL_S = 3.0
TIMEOUT_S = 5.0
POLL_S = 0.2

MODULE_ADDRESS = 2
MODULE_CHANNEL = 0
TEST_VOLTAGE_V = 5.0
TEST_VOLTAGES_V = [1.0, 2.0, 3.0, 4.0]


def show(label, value):
    print(label)
    print(value)
    print()


## 1. Simple Sequence

Recommended high-level sequence: instantiate, initialize, set one voltage,
read back the module snapshot, then shutdown.


In [ ]:
ampr_simple = AMPR(
    device_id=DEVICE_ID,
    com=COM_PORT,
    baudrate=BAUDRATE,
    hk_interval_s=HK_INTERVAL_S,
)

try:
    ampr_simple.initialize(timeout_s=TIMEOUT_S, poll_s=POLL_S)
    modules = ampr_simple.scan_modules()
    print(f"Detected modules: {modules}")

    module_address = next(iter(modules), MODULE_ADDRESS)
    ampr_simple.set_module_voltage(module_address, MODULE_CHANNEL, TEST_VOLTAGE_V)
    show(
        "ampr_simple.get_module_voltages(module_address)",
        ampr_simple.get_module_voltages(module_address),
    )
finally:
    ampr_simple.shutdown()
    ampr_simple.close()


## 2. Connected Session

Use a dedicated instance to exercise connected-state wrapper methods one by one.
`connect()` is the intended entry point here.


In [ ]:
ampr = AMPR(
    device_id=f"{DEVICE_ID}_connected",
    com=COM_PORT,
    baudrate=BAUDRATE,
    hk_interval_s=HK_INTERVAL_S,
)


In [ ]:
show("ampr.get_status()", ampr.get_status())


In [ ]:
show("ampr.connect(timeout_s=TIMEOUT_S)", ampr.connect(timeout_s=TIMEOUT_S))


In [ ]:
show("ampr.get_status()", ampr.get_status())


In [ ]:
show("ampr.format_status(ampr.NO_ERR)", ampr.format_status(ampr.NO_ERR))


In [ ]:
show("ampr.describe_error(ampr.NO_ERR)", ampr.describe_error(ampr.NO_ERR))


In [ ]:
show("ampr.get_sw_version()", ampr.get_sw_version())


In [ ]:
show("ampr.get_product_id()", ampr.get_product_id())


In [ ]:
show("ampr.get_product_no()", ampr.get_product_no())


In [ ]:
show("ampr.get_manuf_date()", ampr.get_manuf_date())


In [ ]:
show("ampr.get_device_type()", ampr.get_device_type())


In [ ]:
show("ampr.get_hw_type()", ampr.get_hw_type())


In [ ]:
show("ampr.get_hw_version()", ampr.get_hw_version())


In [ ]:
show("ampr.get_fw_version()", ampr.get_fw_version())


In [ ]:
show("ampr.get_fw_date()", ampr.get_fw_date())


In [ ]:
show("ampr.get_uptime()", ampr.get_uptime())


In [ ]:
show("ampr.get_optime()", ampr.get_optime())


In [ ]:
show("ampr.get_cpu_data()", ampr.get_cpu_data())


In [ ]:
show("ampr.get_housekeeping()", ampr.get_housekeeping())


In [ ]:
show("ampr.get_state()", ampr.get_state())


In [ ]:
show("ampr.get_device_state()", ampr.get_device_state())


In [ ]:
show("ampr.get_voltage_state()", ampr.get_voltage_state())


In [ ]:
show("ampr.get_temperature_state()", ampr.get_temperature_state())


In [ ]:
show("ampr.get_interlock_state()", ampr.get_interlock_state())


In [ ]:
show("ampr.get_inputs()", ampr.get_inputs())


In [ ]:
show("ampr.get_sync_control()", ampr.get_sync_control())


In [ ]:
show("ampr.get_fan_data()", ampr.get_fan_data())


In [ ]:
show("ampr.get_led_data()", ampr.get_led_data())


In [ ]:
show("ampr.get_module_presence()", ampr.get_module_presence())


In [ ]:
show("ampr.update_module_presence()", ampr.update_module_presence())


In [ ]:
show("ampr.scan_all_modules()", ampr.scan_all_modules())


In [ ]:
show("ampr.scan_modules()", ampr.scan_modules())


In [ ]:
show("ampr.get_scanned_module_state()", ampr.get_scanned_module_state())


In [ ]:
show("ampr.disconnect()", ampr.disconnect())


In [ ]:
ampr.close()


## 3. Initialized Session

Use a fresh initialized instance for module-control and housekeeping helpers.
Run cleanup cells at the end of the section before reconnecting elsewhere.


In [ ]:
ampr_oper = AMPR(
    device_id=f"{DEVICE_ID}_oper",
    com=COM_PORT,
    baudrate=BAUDRATE,
    hk_interval_s=HK_INTERVAL_S,
)
ampr_oper.initialize(timeout_s=TIMEOUT_S, poll_s=POLL_S)

module_candidates = ampr_oper.scan_modules()
print(f"Detected modules: {module_candidates}")
MODULE_ADDRESS = next(iter(module_candidates), MODULE_ADDRESS)


In [ ]:
show("ampr_oper.get_status()", ampr_oper.get_status())


In [ ]:
show("ampr_oper.enable_psu(True)", ampr_oper.enable_psu(True))


In [ ]:
show("ampr_oper.get_module_info(MODULE_ADDRESS)", ampr_oper.get_module_info(MODULE_ADDRESS))


In [ ]:
show("ampr_oper.get_scanned_module_params(MODULE_ADDRESS)", ampr_oper.get_scanned_module_params(MODULE_ADDRESS))


In [ ]:
show("ampr_oper.get_module_fw_version(MODULE_ADDRESS)", ampr_oper.get_module_fw_version(MODULE_ADDRESS))


In [ ]:
show("ampr_oper.get_module_product_no(MODULE_ADDRESS)", ampr_oper.get_module_product_no(MODULE_ADDRESS))


In [ ]:
show("ampr_oper.get_module_hw_type(MODULE_ADDRESS)", ampr_oper.get_module_hw_type(MODULE_ADDRESS))


In [ ]:
show("ampr_oper.get_module_hw_version(MODULE_ADDRESS)", ampr_oper.get_module_hw_version(MODULE_ADDRESS))


In [ ]:
show("ampr_oper.get_module_housekeeping(MODULE_ADDRESS)", ampr_oper.get_module_housekeeping(MODULE_ADDRESS))


In [ ]:
show("ampr_oper.get_module_state(MODULE_ADDRESS)", ampr_oper.get_module_state(MODULE_ADDRESS))


In [ ]:
show("ampr_oper.get_module_voltage_setpoint(MODULE_ADDRESS, MODULE_CHANNEL)", ampr_oper.get_module_voltage_setpoint(MODULE_ADDRESS, MODULE_CHANNEL))


In [ ]:
show("ampr_oper.get_module_voltage_measured(MODULE_ADDRESS, MODULE_CHANNEL)", ampr_oper.get_module_voltage_measured(MODULE_ADDRESS, MODULE_CHANNEL))


In [ ]:
show("ampr_oper.get_all_module_voltages(MODULE_ADDRESS)", ampr_oper.get_all_module_voltages(MODULE_ADDRESS))


In [ ]:
show("ampr_oper.get_all_module_voltage_measured(MODULE_ADDRESS)", ampr_oper.get_all_module_voltage_measured(MODULE_ADDRESS))


In [ ]:
show("ampr_oper.get_module_voltages(MODULE_ADDRESS)", ampr_oper.get_module_voltages(MODULE_ADDRESS))


In [ ]:
ampr_oper.set_module_voltage(MODULE_ADDRESS, MODULE_CHANNEL, TEST_VOLTAGE_V)


In [ ]:
show("ampr_oper.get_module_voltage_setpoint(MODULE_ADDRESS, MODULE_CHANNEL)", ampr_oper.get_module_voltage_setpoint(MODULE_ADDRESS, MODULE_CHANNEL))


In [ ]:
ampr_oper.set_module_voltages(MODULE_ADDRESS, TEST_VOLTAGES_V)


In [ ]:
show("ampr_oper.get_module_voltages(MODULE_ADDRESS)", ampr_oper.get_module_voltages(MODULE_ADDRESS))


In [ ]:
show("ampr_oper.hk_monitor()", ampr_oper.hk_monitor())


In [ ]:
show("ampr_oper.do_housekeeping_cycle()", ampr_oper.do_housekeeping_cycle())


In [ ]:
show("ampr_oper.start_housekeeping(interval_s=HK_INTERVAL_S)", ampr_oper.start_housekeeping(interval_s=HK_INTERVAL_S))


In [ ]:
time.sleep(1.0)


In [ ]:
show("ampr_oper.stop_housekeeping()", ampr_oper.stop_housekeeping())


### Optional / Disruptive AMPR Calls

The cells below intentionally change controller state more aggressively.
Uncomment them only when that behavior is desired on the hardware under test.


In [ ]:
# show("ampr_oper.set_scanned_module_state()", ampr_oper.set_scanned_module_state())


In [ ]:
# show("ampr_oper.rescan_modules()", ampr_oper.rescan_modules())


In [ ]:
# show("ampr_oper.rescan_module(MODULE_ADDRESS)", ampr_oper.rescan_module(MODULE_ADDRESS))


In [ ]:
# show("ampr_oper.restart_module(MODULE_ADDRESS)", ampr_oper.restart_module(MODULE_ADDRESS))


In [ ]:
# show("ampr_oper.set_sync_control(False, False, False)", ampr_oper.set_sync_control(False, False, False))


In [ ]:
# show("ampr_oper.set_interlock_state(0)", ampr_oper.set_interlock_state(0))


In [ ]:
# show("ampr_oper.restart()", ampr_oper.restart())


In [ ]:
ampr_oper.shutdown()


In [ ]:
ampr_oper.close()


## 4. Low-Level Transport Session

Dedicated instance for the explicit transport primitives. Do not mix this
section with `connect()` / `initialize()` on the same object.


In [ ]:
ampr_low = AMPR(
    device_id=f"{DEVICE_ID}_low",
    com=COM_PORT,
    baudrate=BAUDRATE,
    hk_interval_s=HK_INTERVAL_S,
)


In [ ]:
show("ampr_low.open_port(COM_PORT)", ampr_low.open_port(COM_PORT))


In [ ]:
show("ampr_low.set_baud_rate(BAUDRATE)", ampr_low.set_baud_rate(BAUDRATE))


In [ ]:
show("ampr_low.purge()", ampr_low.purge())


In [ ]:
show("ampr_low.device_purge()", ampr_low.device_purge())


In [ ]:
show("ampr_low.get_buffer_state()", ampr_low.get_buffer_state())


In [ ]:
show("ampr_low.close_port()", ampr_low.close_port())


In [ ]:
ampr_low.close()
